In [ ]:
import torch
import json
import os
import sys
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video

# ==================================================================================
# 1. CONFIGURATION
# ==================================================================================
JSON_FILE_PATH = "eeg_results_text.json"
OUTPUT_DIR = "output_videos_light"

# --- SWITCHING TO MODELSCOPE (Lightweight) ---
# This model is 1.7B parameters (vs Zeroscope which is heavier)
# Native resolution is 256x256, which is very easy for 8GB VRAM.
HF_MODEL_ID = "damo-vilab/text-to-video-ms-1.7b"

LOCAL_MODELS_DIR = "/home/poorna/models"
LOCAL_MODEL_PATH = os.path.join(LOCAL_MODELS_DIR, "text-to-video-ms-1.7b")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running Video Generation on: {device}")

# ==================================================================================
# 2. LOAD MODEL SAFELY
# ==================================================================================
print(f"[INFO] Loading Model: {HF_MODEL_ID}")
print("       (If this is the first run, it will download approx 2-3GB)")

try:
    # We load directly. Diffusers handles the cache automatically in standard paths.
    # If you want to force it to your specific folder, we use cache_dir
    pipe = DiffusionPipeline.from_pretrained(
        HF_MODEL_ID,
        torch_dtype=torch.float16,
        variant="fp16",
        cache_dir=LOCAL_MODELS_DIR
    )
    
    # --- CRITICAL OPTIMIZATIONS FOR 8GB VRAM ---
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_model_cpu_offload() # Takes VRAM usage down to ~4GB
    pipe.enable_vae_slicing()       # Prevents memory spikes during decoding
    
except Exception as e:
    print(f"\n[ERROR] Failed to load model: {e}")
    sys.exit(1)

# ==================================================================================
# 3. INTERACTIVE LOOP
# ==================================================================================
if __name__ == "__main__":
    if not os.path.exists(JSON_FILE_PATH):
        print(f"Error: Could not find {JSON_FILE_PATH}.")
        sys.exit(1)
        
    print("Loading JSON data...")
    with open(JSON_FILE_PATH, 'r') as f:
        data = json.load(f)
    data_lookup = {item['index']: item for item in data}

    while True:
        user_input = input("\n[Small Video Gen] Enter Index (or 'q' to quit): ")
        if user_input.lower() == 'q': break
        
        try:
            target_idx = int(user_input)
        except ValueError:
            continue

        entry = data_lookup.get(target_idx)
        if not entry:
            print("Index not found.")
            continue
            
        prompt = entry['predicted_text']
        gt_text = entry['ground_truth_text']
        
        print(f"\n--- Sample {target_idx} ---")
        print(f"Ground Truth: '{gt_text}'")
        print(f"Generating:   '{prompt}'")
        
        try:
            # Generate Video (Native resolution 256x256)
            video_frames = pipe(
                prompt=prompt,
                num_frames=16,          # 16 frames is standard for this model
                num_inference_steps=25  # 25 steps is enough for draft quality
            ).frames[0]

            # Save
            safe_prompt = "".join([c if c.isalnum() else "_" for c in prompt])[:20]
            filename = f"idx_{target_idx}_{safe_prompt}.mp4"
            save_path = os.path.join(OUTPUT_DIR, filename)
            
            export_to_video(video_frames, save_path, fps=8)
            print(f"SUCCESS! Saved to: {save_path}")
            
        except Exception as e:
            print(f"Error: {e}")
            print("Try closing your web browser/other apps to free up GPU memory.")

Running Video Generation on: cuda
[INFO] Loading Model: damo-vilab/text-to-video-ms-1.7b
       (If this is the first run, it will download approx 2-3GB)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Loading JSON data...



[Small Video Gen] Enter Index (or 'q' to quit):  4


...

SUCCESS! Saved to: output_videos_light/idx_4_a_city_with_tall_bui.mp4



[Small Video Gen] Enter Index (or 'q' to quit):  9



--- Sample 9 ---
Ground Truth: 'a beach with a cloudy sky and a beach'
Generating:   'a waterfall in the middle of a river'


  0%|          | 0/25 [00:00<?, ?it/s]

SUCCESS! Saved to: output_videos_light/idx_9_a_waterfall_in_the_m.mp4
